# 02_Finetune_Gemma3_Multimodal_From_HF_Dataset

이 노트북은 Hugging Face에 업로드한 통합 데이터셋을 내려받아,
Gemma 3 멀티모달 포맷으로 변환한 뒤 QLoRA로 학습합니다.

- 텍스트-only 샘플도 같은 모델로 학습
- 이미지+텍스트 샘플도 같은 모델로 학습
- 배치 크기는 기본적으로 1로 두어 모달 혼합 충돌을 줄입니다.

In [1]:
import os
import random
from datetime import datetime

import pandas as pd
from datasets import load_dataset

import torch
from peft import LoraConfig
from transformers import (
    AutoProcessor, AutoModelForImageTextToText,
    BitsAndBytesConfig, TrainerCallback
)
from trl import SFTTrainer, SFTConfig

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.font_manager as fm

from PIL import Image
from dotenv import load_dotenv
import huggingface_hub
import mysql.connector

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    print('GPU =', torch.cuda.get_device_name(0))
print('torch =', torch.__version__)

GPU = NVIDIA B200
torch = 2.12.1+cu130


In [2]:
def setKoreanFont():
    """matplotlib 한글 폰트 설정 (Windows: 맑은 고딕 우선 탐색)"""
    fontCandidates = ['Malgun Gothic', 'NanumGothic', 'NanumBarunGothic', 'AppleGothic', 'DejaVu Sans']
    allFonts       = fm.fontManager.ttflist
    availableFonts = []
    for i in range(0, len(allFonts)):
        availableFonts.append(allFonts[i].name)

    selectedFont = None
    for i in range(0, len(fontCandidates)):
        if fontCandidates[i] in availableFonts:
            selectedFont = fontCandidates[i]
            break

    if selectedFont is None:
        selectedFont = 'DejaVu Sans'
        print(f'[경고] 한글 폰트 없음, 대체 사용: {selectedFont}')
    else:
        print(f'한글 폰트 설정: {selectedFont}')

    matplotlib.rc('font', family=selectedFont)
    matplotlib.rcParams['axes.unicode_minus'] = False

setKoreanFont()

한글 폰트 설정: NanumGothic


In [3]:
# .env 파일 로드 (override=True: .env 수정 후 재실행 시 항상 반영)
load_dotenv(override=True)
hf_token = os.getenv("HF_TOKEN", "")

In [4]:
if hf_token and hf_token not in ('YOUR_HF_TOKEN', ''):
    try:
        huggingface_hub.login(hf_token)
    except Exception as e:
        print(f"[경고] HuggingFace 로그인 실패: {e}")
        print("[경고] .env의 HF_TOKEN을 확인하세요. 공개 데이터셋만 접근 가능합니다.")
else:
    print("[경고] HF_TOKEN이 설정되지 않았습니다. 공개 데이터셋만 접근 가능합니다.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [ ]:
# ===== .env 전체 설정값 =====
DATASET_REPO = os.getenv("DATASET_REPO", "hyokwan/ocr_dataset")

BASE_MODEL   = os.getenv("BASE_MODEL",   "google/gemma-3-4b-it")

OUTPUT_BASE_DIR = os.getenv("OUTPUT_BASE_DIR", "./models")
OUTPUT_DIR      = os.path.join(
    OUTPUT_BASE_DIR,
    f"gemma3_multimodal_lora_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
)
TENSORBOARD_DIR = os.path.join(OUTPUT_DIR, "tensorboard")

_trainSamples     = os.getenv("MAX_TRAIN_SAMPLES", "")
_evalSamples      = os.getenv("MAX_EVAL_SAMPLES",  "")
MAX_TRAIN_SAMPLES = int(_trainSamples) if _trainSamples.strip() else None
MAX_EVAL_SAMPLES  = int(_evalSamples)  if _evalSamples.strip()  else None

MAX_SEQ_LENGTH              = int(os.getenv("MAX_SEQ_LENGTH",              768))
PER_DEVICE_TRAIN_BATCH_SIZE = int(os.getenv("PER_DEVICE_TRAIN_BATCH_SIZE", 1))
PER_DEVICE_EVAL_BATCH_SIZE  = int(os.getenv("PER_DEVICE_EVAL_BATCH_SIZE",  1))
GRAD_ACCUM                  = int(os.getenv("GRAD_ACCUM",                  4))
NUM_EPOCHS                  = int(os.getenv("NUM_EPOCHS",                  5))
LEARNING_RATE               = float(os.getenv("LEARNING_RATE",             2e-4))
LOGGING_STEPS               = int(os.getenv("LOGGING_STEPS",               10))
SAVE_STEPS                  = int(os.getenv("SAVE_STEPS",                  100))

LORA_R       = int(os.getenv("LORA_R",       16))
LORA_ALPHA   = int(os.getenv("LORA_ALPHA",   32))
LORA_DROPOUT = float(os.getenv("LORA_DROPOUT", 0.05))

DB_CFG = {
    'host':     os.getenv("DB_HOST",     "localhost"),
    'port':     int(os.getenv("DB_PORT", 3306)),
    'database': os.getenv("DB_NAME",     ""),
    'user':     os.getenv("DB_USER",     ""),
    'password': os.getenv("DB_PASSWORD", ""),
}

print(f"데이터셋    : {DATASET_REPO}")
print(f"베이스 모델 : {BASE_MODEL}")
print(f"출력 경로   : {OUTPUT_DIR}")
print(f"TensorBoard : {TENSORBOARD_DIR}")
print(f"에폭 {NUM_EPOCHS} | LR {LEARNING_RATE} | LoRA r/α {LORA_R}/{LORA_ALPHA}")

데이터셋    : hyokwan/multi_modal_sample
베이스 모델 : google/gemma-3-4b-it
출력 경로   : ./models/gemma3_multimodal_lora_20260622_141010
TensorBoard : ./models/gemma3_multimodal_lora_20260622_141010/tensorboard
에폭 5 | LR 0.0002 | LoRA r/α 16/32


In [ ]:
def getDbConn(dbCfg):
    """MySQL 연결 객체 반환"""
    return mysql.connector.connect(**dbCfg)


def getNextSimSeq(dbCfg):
    """DB에서 다음 시뮬레이션 시퀀스 번호 조회 (MAX + 1)"""
    try:
        conn   = getDbConn(dbCfg)
        cursor = conn.cursor()
        cursor.execute("SELECT COALESCE(MAX(sim_seq), 0) + 1 FROM simulation_runs")
        simSeq = cursor.fetchone()[0]
        cursor.close()
        conn.close()
        return int(simSeq)
    except Exception as e:
        print(f"[DB] sim_seq 조회 실패: {e}")
        return 1


def insertRunStart(simSeq, dbCfg):
    """학습 시작 시 simulation_runs에 레코드 삽입 (status=running)"""
    try:
        conn   = getDbConn(dbCfg)
        cursor = conn.cursor()
        cursor.execute(
            """
            INSERT INTO simulation_runs (
                sim_seq, status, run_timestamp,
                model_name, dataset_repo, output_dir, tensorboard_dir,
                num_epochs, learning_rate, lora_r, lora_alpha, lora_dropout,
                max_seq_length, grad_accum, batch_size, logging_steps, save_steps
            ) VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
            """,
            (
                simSeq, 'running', datetime.now(),
                BASE_MODEL, DATASET_REPO, OUTPUT_DIR, TENSORBOARD_DIR,
                NUM_EPOCHS, LEARNING_RATE, LORA_R, LORA_ALPHA, LORA_DROPOUT,
                MAX_SEQ_LENGTH, GRAD_ACCUM, PER_DEVICE_TRAIN_BATCH_SIZE,
                LOGGING_STEPS, SAVE_STEPS,
            )
        )
        conn.commit()
        cursor.close()
        conn.close()
        print(f"[DB] 시뮬레이션 #{simSeq} 시작 기록 완료")
    except Exception as e:
        print({"success": False, "message": f"insertRunStart 실패: {e}"})


def updateRunEnd(simSeq, trainResult, dbCfg, status='completed', errorMsg=None, bestEvalLoss=None):
    """학습 완료/실패 시 simulation_runs 레코드 업데이트 — train_loss에 best eval_loss 저장"""
    try:
        conn   = getDbConn(dbCfg)
        cursor = conn.cursor()
        if trainResult is not None:
            metrics = trainResult.metrics
            cursor.execute(
                """
                UPDATE simulation_runs SET
                    status=%s, end_timestamp=%s,
                    train_loss=%s, runtime_sec=%s, samples_per_sec=%s, global_step=%s
                WHERE sim_seq=%s
                """,
                (
                    status, datetime.now(),
                    bestEvalLoss,
                    metrics.get('train_runtime'),
                    metrics.get('train_samples_per_second'),
                    trainResult.global_step,
                    simSeq,
                )
            )
        else:
            cursor.execute(
                "UPDATE simulation_runs SET status=%s, end_timestamp=%s, error_message=%s WHERE sim_seq=%s",
                (status, datetime.now(), errorMsg, simSeq)
            )
        conn.commit()
        cursor.close()
        conn.close()
        print(f"[DB] 시뮬레이션 #{simSeq} 종료 기록 완료 (status={status})")
    except Exception as e:
        print({"success": False, "message": f"updateRunEnd 실패: {e}"})


class DbLogCallback(TrainerCallback):
    """학습 스텝별 손실을 simulation_logs 테이블에 실시간 저장"""

    def __init__(self, simSeq, dbCfg):
        """콜백 초기화"""
        self.simSeq = simSeq
        self.dbCfg  = dbCfg

    def on_log(self, args, state, control, logs=None, **kwargs):
        """로깅 이벤트 발생 시 DB에 스텝 로그 저장"""
        if logs is None or 'loss' not in logs:
            return
        try:
            conn   = getDbConn(self.dbCfg)
            cursor = conn.cursor()
            cursor.execute(
                """
                INSERT INTO simulation_logs (sim_seq, step, loss, learning_rate, epoch, log_timestamp)
                VALUES (%s, %s, %s, %s, %s, %s)
                """,
                (
                    self.simSeq,
                    state.global_step,
                    logs.get('loss'),
                    logs.get('learning_rate'),
                    logs.get('epoch'),
                    datetime.now(),
                )
            )
            conn.commit()
            cursor.close()
            conn.close()
        except Exception as e:
            print(f"[DB] 스텝 로그 저장 실패 (step={state.global_step}): {e}")


SIM_SEQ = getNextSimSeq(DB_CFG)
insertRunStart(SIM_SEQ, DB_CFG)
print(f"====== 시뮬레이션 #{SIM_SEQ} 시작 ======")

### 데이터셋 불러오기

In [7]:
ds = load_dataset(DATASET_REPO)
# 첫 번째 split 사용
dataset = ds[list(ds.keys())[0]]
# 8:2 분할
split_ds = dataset.train_test_split(
    test_size=0.2,
    seed=42
)

train_ds = split_ds["train"]
eval_ds = split_ds["test"]

if MAX_TRAIN_SAMPLES:
    train_ds = train_ds.select(
        range(min(MAX_TRAIN_SAMPLES, len(train_ds)))
    )

if MAX_EVAL_SAMPLES:
    eval_ds = eval_ds.select(
        range(min(MAX_EVAL_SAMPLES, len(eval_ds)))
    )

print(train_ds)
print(eval_ds)
print(train_ds[0])


Dataset({
    features: ['modality', 'image', 'instruction', 'input', 'output', 'source', 'label'],
    num_rows: 44
})
Dataset({
    features: ['modality', 'image', 'instruction', 'input', 'output', 'source', 'label'],
    num_rows: 11
})
{'modality': 'image_text', 'image': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=896x1200 at 0x7A667E6B4F80>, 'instruction': '이미지를 보고 필라테스 동작명을 분류하고 한 줄 설명을 제공하세요.', 'input': '동작명과 간단한 설명을 한국어로 답하세요.', 'output': '동작명: bridge\n설명: 이 이미지는 bridge 동작 예시입니다.', 'source': 'pilates_dataset', 'label': 'bridge'}


# 3. 환경 및 최적화 설정 (4비트 양자화)

In [8]:
# 현재 사용 중인 GPU의 주요 아키텍처 버전을 반환 8버전 이상 시 bfloat16 활용
# NVIDIA Ampere 아키텍처 이상 시에만 처리
if torch.cuda.get_device_capability()[0] >= 8:
    # 고속 attention 메커니즘을 구현하는 라이브러리
    attn_implementation = "flash_attention_2"
    torch_dtype = torch.bfloat16
else:
    attn_implementation = "eager"
    torch_dtype = torch.float16

# BitsAndBytesConfig 객체활용 양자화 설정
quant_config = BitsAndBytesConfig(
    # 모델을 4비트 양자화하여 로드할지 여부 결정
    load_in_4bit=True,
    # 양자화 방법 (nf4: Non-Uniform Quantization, "nf4","fp16 등))
    bnb_4bit_quant_type="nf4",
    # (4비트 양자화 시 사용할 데이터 타입, "torch.float16, bfloat16, float32 등)
    bnb_4bit_compute_dtype=torch_dtype,
    # 이중 양자화 사용여부 (이중 양자화는 양자화 과정에서 정밀도 높이기 위해 활용, 대신 더 연산은 복잡)
    bnb_4bit_use_double_quant=False
)

# 4. 베이스 모델 불러오기

In [ ]:
# 멀티모달 모델 로드 (이미지 + 텍스트 입력 처리 가능)
model = AutoModelForImageTextToText.from_pretrained(
    BASE_MODEL,
    # quantization_config=quant_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
    attn_implementation='eager',
    trust_remote_code=True,
    token = hf_token
)

# 멀티모달 모델 로드 (이미지 + 텍스트 입력 처리 가능)
processor = AutoProcessor.from_pretrained(BASE_MODEL, trust_remote_code=True,token = hf_token)

# processor 내부에서 tokenizer만 따로 사용
tokenizer = processor.tokenizer
# pad_token 없으면 eos_token으로 설정 # 학습 시 길이 같아야 함 문장1-안녕하세요 문장2 안녕 -> 안녕[pad][pad]
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token

print('pad_token_id =', tokenizer.pad_token_id)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


pad_token_id = 0


In [10]:
SYSTEM_MESSAGE = '당신은 멀티모달 도우미입니다. 텍스트 질문에는 정확히 답하고, 이미지가 주어지면 이미지를 보고 답하세요.'

def buildMessages(example):
    """샘플 데이터를 모달리티(텍스트/이미지+텍스트)에 따라 모델 메시지 구조로 변환"""
    instruction = str(example.get('instruction', '')).strip()
    userInput   = str(example.get('input', '')).strip()
    output      = str(example.get('output', '')).strip()
    modality    = str(example.get('modality', 'text_only')).strip()

    if instruction and userInput:
        userText = f"{instruction}\n\n[입력]\n{userInput}"
    else:
        userText = instruction or userInput
    if not userText:
        userText = '주어진 정보를 바탕으로 답하세요.'

    userContent = []
    if modality == 'image_text' and example.get('image') is not None:
        userContent.append({'type': 'image', 'image': example['image']})
    userContent.append({'type': 'text', 'text': userText})

    messages = [
        {'role': 'system',    'content': [{'type': 'text', 'text': SYSTEM_MESSAGE}]},
        {'role': 'user',      'content': userContent},
        {'role': 'assistant', 'content': [{'type': 'text', 'text': output}]},
    ]
    return messages

In [11]:
# 소규모 샘플 확인
sample_rows = []
for ex in train_ds.select(range(min(4, len(train_ds)))):
    sample_rows.append({
        'modality': ex['modality'],
        'instruction': ex['instruction'][:80],
        'input': ex['input'][:80],
        'output': ex['output'][:80],
        'label': ex['label'],
        'image_present': ex['image'] is not None,
    })
pd.DataFrame(sample_rows)

,modality,instruction,input,output,label,image_present
0,image_text,이미지를 보고 필라테스 동작명을 분류하고 한 줄 설명을 제공하세요.,동작명과 간단한 설명을 한국어로 답하세요.,동작명: bridge\n설명: 이 이미지는 bridge 동작 예시입니다.,bridge,True
1,image_text,이미지를 보고 필라테스 동작명을 분류하고 한 줄 설명을 제공하세요.,동작명과 간단한 설명을 한국어로 답하세요.,동작명: hundred\n설명: 이 이미지는 hundred 동작 예시입니다.,hundred,True
2,image_text,이미지를 보고 필라테스 동작명을 분류하고 한 줄 설명을 제공하세요.,동작명과 간단한 설명을 한국어로 답하세요.,동작명: bridge\n설명: 이 이미지는 bridge 동작 예시입니다.,bridge,True
3,image_text,이미지를 보고 필라테스 동작명을 분류하고 한 줄 설명을 제공하세요.,동작명과 간단한 설명을 한국어로 답하세요.,동작명: teaser\n설명: 이 이미지는 teaser 동작 예시입니다.,teaser,True


In [ ]:
def _mergeMixedBatch(imgBatch, txtBatch, imgIdx, txtIdx, totalSize):
    """혼합 배치의 이미지/텍스트 서브배치를 원본 순서로 병합"""
    imgSeqLen  = imgBatch["input_ids"].shape[1]
    txtSeqLen  = txtBatch["input_ids"].shape[1]
    maxSeqLen  = max(imgSeqLen, txtSeqLen)
    padId      = processor.tokenizer.pad_token_id or 0
    seqKeys    = ["input_ids", "attention_mask", "token_type_ids"]
    seqPadMap  = {"input_ids": padId, "attention_mask": 0, "token_type_ids": 0}
    mergedDict = {}

    for ki in range(0, len(seqKeys)):
        key       = seqKeys[ki]
        padVal    = seqPadMap[key]
        imgTensor = imgBatch.get(key)
        txtTensor = txtBatch.get(key)
        if imgTensor is None and txtTensor is None:
            continue
        dtype = (imgTensor if imgTensor is not None else txtTensor).dtype
        if imgTensor is not None:
            imgPadded = torch.nn.functional.pad(imgTensor, (0, maxSeqLen - imgSeqLen), value=padVal)
        else:
            imgPadded = torch.full((len(imgIdx), maxSeqLen), padVal, dtype=dtype)
        if txtTensor is not None:
            txtPadded = torch.nn.functional.pad(txtTensor, (0, maxSeqLen - txtSeqLen), value=padVal)
        else:
            txtPadded = torch.full((len(txtIdx), maxSeqLen), padVal, dtype=dtype)
        outTensor = torch.full((totalSize, maxSeqLen), padVal, dtype=dtype)
        for k in range(0, len(imgIdx)):
            outTensor[imgIdx[k]] = imgPadded[k]
        for k in range(0, len(txtIdx)):
            outTensor[txtIdx[k]] = txtPadded[k]
        mergedDict[key] = outTensor

    if "pixel_values" in imgBatch:
        mergedDict["pixel_values"] = imgBatch["pixel_values"]

    return mergedDict


def collate_fn(examples):
    """텍스트 + 이미지 데이터를 모델 학습용 배치로 변환 및 불필요 토큰 loss 제외"""
    allTexts     = []
    nestedImages = []  # 샘플당 [img] 또는 None (중첩 리스트 구조)
    hasAnyImage  = False

    for i in range(0, len(examples)):
        example  = examples[i]
        messages = buildMessages(example)
        text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        allTexts.append(text)

        sampleImage = None
        userContent = messages[1]["content"]
        for j in range(0, len(userContent)):
            if userContent[j]["type"] == "image":
                sampleImage = userContent[j]["image"]
                hasAnyImage = True
                break

        if sampleImage is not None:
            nestedImages.append([sampleImage])  # 샘플당 [img] 중첩 — processor가 배치 경계를 인식하도록
        else:
            nestedImages.append(None)

    if not hasAnyImage:
        # 전부 text-only
        batch = processor(text=allTexts, return_tensors="pt", padding=True)

    else:
        allHaveImage = True
        for i in range(0, len(nestedImages)):
            if nestedImages[i] is None:
                allHaveImage = False
                break

        if allHaveImage:
            # 전부 image-text → 중첩 리스트로 한 번에 처리
            # [[img1],[img2],...] 구조여야 processor가 len==batchSize로 인식
            batch = processor(text=allTexts, images=nestedImages, return_tensors="pt", padding=True)

        else:
            # 혼합 배치 → 이미지/텍스트 각각 분리 처리 후 원본 순서로 병합
            imgIdx = []
            txtIdx = []
            for i in range(0, len(nestedImages)):
                if nestedImages[i] is not None:
                    imgIdx.append(i)
                else:
                    txtIdx.append(i)

            imgTexts  = []
            imgImages = []
            for k in range(0, len(imgIdx)):
                imgTexts.append(allTexts[imgIdx[k]])
                imgImages.append(nestedImages[imgIdx[k]])

            txtTexts = []
            for k in range(0, len(txtIdx)):
                txtTexts.append(allTexts[txtIdx[k]])

            imgBatch = processor(text=imgTexts, images=imgImages, return_tensors="pt", padding=True)
            txtBatch = processor(text=txtTexts, return_tensors="pt", padding=True)
            batch    = _mergeMixedBatch(imgBatch, txtBatch, imgIdx, txtIdx, len(allTexts))

    labels     = batch["input_ids"].clone()
    padTokenId = processor.tokenizer.pad_token_id
    if padTokenId is not None:
        labels[labels == padTokenId] = -100

    imageTokenId = getattr(model.config, "image_token_id", None)
    if imageTokenId is not None:
        labels[labels == imageTokenId] = -100

    labels[labels == 262144] = -100
    batch["labels"] = labels
    return batch

## 5. LoRA 및 Trainer 설정

In [ ]:
peft_params = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules='all-linear',
)

sft_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    logging_dir=TENSORBOARD_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    learning_rate=LEARNING_RATE,
    weight_decay=0.001,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    lr_scheduler_type="constant",
    logging_steps=LOGGING_STEPS,
    report_to='tensorboard',
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    bf16=torch.cuda.is_available(),
    remove_unused_columns=False,
    dataset_kwargs={"skip_prepare_dataset": True},
)

dbCallback = DbLogCallback(SIM_SEQ, DB_CFG)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    peft_config=peft_params,
    processing_class=processor,
    data_collator=collate_fn,
    args=sft_args,
    callbacks=[dbCallback],
)

print(f"TensorBoard: tensorboard --logdir {TENSORBOARD_DIR}")

In [ ]:
train_result = trainer.train()

bestEvalLoss = trainer.state.best_metric
print(f"Best eval_loss: {bestEvalLoss}")

updateRunEnd(SIM_SEQ, train_result, DB_CFG, status='completed', bestEvalLoss=bestEvalLoss)

In [15]:
def infer_unified(question, image=None, system=SYSTEM_MESSAGE, max_new_tokens=128, temperature=0.2):
    # 시스템 / 사용자 메시지 구조 생성 (멀티모달 형식)
    messages = [
        {'role': 'system', 'content': [{'type': 'text', 'text': system}]},
        {'role': 'user', 'content': []},
    ]

    # 이미지가 있으면 user 메시지에 추가
    if image is not None:
        messages[1]['content'].append({'type': 'image', 'image': image})

    # 질문 텍스트 추가
    messages[1]['content'].append({'type': 'text', 'text': question})

    # chat template 적용 → 모델 입력용 프롬프트 생성
    prompt = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True  # 답변 생성 유도
    )

    # 이미지 포함 여부에 따라 processor 호출 방식 분기
    if image is not None:
        inputs = processor(
            text=[prompt],
            images=[[image]],  # 배치 형태 (list of list)
            return_tensors='pt',
            padding=True
        )
    else:
        inputs = processor(
            text=[prompt],
            return_tensors='pt',
            padding=True
        )

    # 모델 device(GPU/CPU)로 이동
    inputs = {
        k: v.to(model.device) if hasattr(v, 'to') else v
        for k, v in inputs.items()
    }

    # 추론 (gradient 비활성화)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0,           # temperature 0이면 greedy
            temperature=max(temperature, 1e-5), # 0 division 방지
        )

    # 입력 프롬프트 이후 생성된 부분만 잘라냄
    answer_ids = outputs[:, inputs['input_ids'].shape[1]:]

    # 토큰 → 텍스트 변환
    return processor.batch_decode(
        answer_ids,
        skip_special_tokens=True
    )[0].strip()

In [47]:
# 텍스트-only 테스트
text_answer = infer_unified(
    question='금융 용어를 쉽게 설명하시오.\n\n[입력]\n기준금리',
    system='당신은 금융 개념을 쉽게 설명하는 도우미입니다.',
    max_new_tokens=96,
    temperature=0.0,
)
print(text_answer)

c:\Users\hkcode\llm_multimodal\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `1e-05` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\Users\hkcode\llm_multimodal\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
c:\Users\hkcode\llm_multimodal\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `64` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
c:\Users\hkcode\llm_multimodal\.venv\Lib\site-package

기준금리는 금융 시장에서 중요한 역할을 하는 기준 금리다. 쉽게 설명하면 다음과 같다.

*   **목표 금리:** 중앙은행이 금융기관에 빌려주는 돈의 금리를 정하는 기준이다.
*   **영향:** 기준금리가 오르면 대출 금리, 예금 금리 등이 모두 상승하고, 반대로 기준금리가 하락하면 금리 인하로 이어질 수 있다.
*   **역


In [48]:
# 텍스트-only 테스트
text_answer = infer_unified(
    question='다음 문장을 코칭 스타일로 바꾸시오.\n\n[입력]\n허리를 펴세요',
    system='당신은 AI 코칭 도우미입니다.',
    max_new_tokens=96,
    temperature=0.0,
)
print(text_answer)

허리를 곧게 세우고 코어에 힘을 주세요.


In [49]:
from PIL import Image

# 이미지 로드
image = Image.open("test_001.png").convert("RGB")

# 질문
question = "이 이미지에 무슨 동작 인가요?"

# 추론 실행
result = infer_unified(question, image=image)

print(result)

누워서 다리를 들어 올려 복부를 강화하는 동작이다.


In [ ]:
# 학습 로그 파싱
rawLogs   = trainer.state.log_history
trainLogs = []
for i in range(0, len(rawLogs)):
    if 'loss' in rawLogs[i] and 'eval_loss' not in rawLogs[i]:
        trainLogs.append(rawLogs[i])

trainSteps  = []
trainLosses = []
trainPpls   = []
lrValues    = []
for i in range(0, len(trainLogs)):
    trainSteps.append(trainLogs[i]['step'])
    trainLosses.append(trainLogs[i]['loss'])
    trainPpls.append(2.718 ** trainLogs[i]['loss'])
    lrValues.append(trainLogs[i].get('learning_rate', 0))

fig = plt.figure(figsize=(14, 10))
fig.suptitle(f"학습 결과\n{OUTPUT_DIR}", fontsize=12, fontweight='bold')
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.35)

ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(trainSteps, trainLosses, color='steelblue', marker='o', markersize=4, linewidth=2)
ax1.set_title("학습 손실 (Training Loss)")
ax1.set_xlabel("스텝 (Step)")
ax1.set_ylabel("손실 (Loss)")
ax1.grid(True, alpha=0.3)

ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(trainSteps, trainPpls, color='darkorange', marker='s', markersize=4, linewidth=2)
ax2.set_title("혼란도 (Perplexity = exp(loss))")
ax2.set_xlabel("스텝 (Step)")
ax2.set_ylabel("PPL")
ax2.grid(True, alpha=0.3)

ax3 = fig.add_subplot(gs[1, 0])
ax3.plot(trainSteps, lrValues, color='green', marker='^', markersize=4, linewidth=2)
ax3.set_title("학습률 (Learning Rate)")
ax3.set_xlabel("스텝 (Step)")
ax3.set_ylabel("LR")
ax3.grid(True, alpha=0.3)

ax4 = fig.add_subplot(gs[1, 1])
ax4.axis('off')

lastMetrics = trainer.state.log_history[-1] if trainer.state.log_history else {}
finalLoss   = trainLosses[-1] if trainLosses else None
finalPpl    = trainPpls[-1]   if trainPpls   else None
trainSec    = lastMetrics.get("train_runtime", None)

tableData = [
    ["모델",          BASE_MODEL.split("/")[-1]],
    ["데이터셋",      DATASET_REPO],
    ["학습 샘플",     len(train_ds)],
    ["에폭",          NUM_EPOCHS],
    ["배치 크기",     PER_DEVICE_TRAIN_BATCH_SIZE],
    ["Grad Accum",    GRAD_ACCUM],
    ["LR",            LEARNING_RATE],
    ["Max Seq Len",   MAX_SEQ_LENGTH],
    ["LoRA r/α",      f"{LORA_R}/{LORA_ALPHA}"],
    ["총 스텝",       trainer.state.global_step],
    ["최종 손실",     f"{finalLoss:.4f}" if finalLoss else "-"],
    ["최종 PPL",      f"{finalPpl:.2f}"  if finalPpl  else "-"],
    ["소요 시간(초)", f"{trainSec:.0f}"  if trainSec  else "-"],
]

tbl = ax4.table(cellText=tableData, colLabels=["항목", "값"],
                loc='center', cellLoc='left')
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1, 1.45)
ax4.set_title("파라미터 & 결과", pad=12)

os.makedirs(OUTPUT_DIR, exist_ok=True)
plt.savefig(os.path.join(OUTPUT_DIR, "train_result.png"), dpi=150, bbox_inches='tight')
plt.show()
print(f"그래프 저장 → {OUTPUT_DIR}/train_result.png")

In [ ]:
# 시뮬레이션 결과 DB 저장 확인
try:
    conn   = mysql.connector.connect(**DB_CFG)
    cursor = conn.cursor(dictionary=True)
    cursor.execute(
        "SELECT sim_seq, status, train_loss, runtime_sec, global_step FROM simulation_runs WHERE sim_seq=%s",
        (SIM_SEQ,)
    )
    row = cursor.fetchone()
    cursor.close()
    conn.close()
    if row:
        print(f"[DB] 시뮬레이션 #{SIM_SEQ} 저장 확인:")
        print(pd.DataFrame([row]).to_string(index=False))
    else:
        print(f"[DB] sim_seq={SIM_SEQ} 조회 결과 없음")
except Exception as e:
    print({"success": False, "message": f"DB 조회 실패: {e}"})

In [52]:
# 학습된 LoRA 어댑터(모델 가중치) 저장
# → 전체 모델이 아니라 "변경된 부분(LoRA)"만 저장됨
trainer.model.save_pretrained(OUTPUT_DIR)

# 멀티모달 전처리기 저장 (tokenizer + image processor 포함)
# → 추론 시 동일한 방식으로 입력 처리하기 위해 필요
processor.save_pretrained(OUTPUT_DIR)

# 저장 완료 로그 출력
print('adapter saved to', OUTPUT_DIR)

adapter saved to ./models/gemma3_multimodal_lora_output_20260611_162018
